# 🎓 Sistema de Predicción de Deserción Estudiantil con Machine Learning

## Foro de Discusión: Componentes de Machine Learning

---

### 📚 Información del Proyecto

| Campo | Descripción |
|-------|-------------|
| **Curso** | Componentes de Machine Learning |
| **Universidad** | UNIMINUTO |
| **Objetivo** | Predecir deserción estudiantil mediante ML |
| **Sector** | Educación Superior |

---

### 🎯 Pregunta Orientadora

> **¿Por qué es importante considerar la cantidad de datos en la aplicación de Machine Learning?**

La cantidad de datos es fundamental porque:

1. **Generalización**: Más datos = mejor capacidad del modelo para generalizar a casos nuevos
2. **Reducción de overfitting**: Datasets pequeños tienden a que el modelo memorice en lugar de aprender
3. **Representatividad estadística**: Se necesitan suficientes ejemplos de cada clase
4. **Validación robusta**: Permite división train/validation/test sin perder representatividad

*Referencias: Barrero Ortiz (2020, p. 16), Véliz Capuñay (2020, p. 45)*

## 📦 1. Configuración del Entorno

Instalamos las dependencias necesarias para el proyecto.

In [ ]:
# Instalación de dependencias (ejecutar solo en Colab)
!pip install -q numpy pandas scikit-learn xgboost imbalanced-learn matplotlib seaborn shap

In [ ]:
# Importación de librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

# XGBoost y SMOTE
import xgboost as xgb
from imblearn.over_sampling import SMOTE

print("✅ Librerías importadas correctamente")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 📊 2. Componente de ENTRADA: Generación de Datos

### Componente de ML: Datos de Entrada (Features)

Los datos de entrada son el combustible del Machine Learning. Según Rothman (2018, pp. 46-58), la calidad y cantidad de datos determina directamente el rendimiento del modelo.

En este caso, generamos datos sintéticos que simulan características reales de estudiantes universitarios colombianos.

In [ ]:
def generate_student_data(n_students=1000, dropout_rate=0.25, random_state=42):
    """
    Genera datos sintéticos de estudiantes universitarios.
    
    COMPONENTES DE ENTRADA:
    - Datos académicos (notas, asistencia, créditos)
    - Datos socioeconómicos (estrato, financiamiento)
    - Datos de comportamiento (uso de plataforma, tutorías)
    - Datos demográficos (edad, distancia)
    """
    np.random.seed(random_state)
    
    # ===============================
    # DATOS ACADÉMICOS
    # ===============================
    
    # Promedio de notas (escala colombiana 0-5)
    # Distribución bimodal: estudiantes buenos y con dificultades
    grades_good = np.random.normal(3.8, 0.4, int(n_students * 0.7))
    grades_low = np.random.normal(2.5, 0.5, int(n_students * 0.3))
    promedio_notas = np.concatenate([grades_good, grades_low])
    np.random.shuffle(promedio_notas)
    promedio_notas = np.clip(promedio_notas[:n_students], 0, 5)
    
    # Asistencia (distribución sesgada hacia alta asistencia)
    asistencia = np.random.beta(5, 1.5, n_students) * 100
    
    # Ratio de créditos aprobados
    ratio_creditos = np.random.beta(4, 1.5, n_students)
    
    # Materias perdidas (distribución geométrica)
    materias_perdidas = np.random.geometric(0.6, n_students) - 1
    
    # ===============================
    # DATOS SOCIOECONÓMICOS
    # ===============================
    
    # Estrato (1-6, distribución típica colombiana)
    estrato = np.random.choice(
        [1, 2, 3, 4, 5, 6],
        size=n_students,
        p=[0.15, 0.30, 0.25, 0.15, 0.10, 0.05]
    )
    
    # Financiamiento
    financiamiento = np.random.choice(
        ['propio', 'icetex', 'beca_completa', 'beca_parcial', 'patrocinio'],
        size=n_students,
        p=[0.40, 0.25, 0.10, 0.15, 0.10]
    )
    
    # Trabaja mientras estudia
    trabaja = np.random.choice([0, 1], n_students, p=[0.45, 0.55])
    
    # Horas de trabajo
    horas_trabajo = np.where(trabaja == 1,
                             np.random.choice([10, 20, 30, 40], n_students),
                             0)
    
    # ===============================
    # DATOS DE COMPORTAMIENTO
    # ===============================
    
    # Horas en plataforma virtual
    horas_plataforma = np.clip(np.random.exponential(5, n_students), 0, 40)
    
    # Interacciones con tutorías
    interacciones_tutorias = np.random.poisson(3, n_students)
    
    # Actividades extracurriculares
    actividades_extra = np.random.poisson(2, n_students)
    
    # ===============================
    # DATOS DEMOGRÁFICOS
    # ===============================
    
    edad = np.clip(np.random.normal(21, 3, n_students), 17, 50).astype(int)
    genero = np.random.choice(['M', 'F'], n_students, p=[0.48, 0.52])
    distancia_campus = np.clip(np.random.exponential(15, n_students), 0, 100)
    semestre = np.random.choice(range(1, 11), n_students,
                                p=[0.20, 0.15, 0.12, 0.10, 0.10, 
                                   0.08, 0.08, 0.07, 0.05, 0.05])
    
    # Crear DataFrame
    df = pd.DataFrame({
        'estudiante_id': range(1, n_students + 1),
        'promedio_notas': promedio_notas.round(2),
        'asistencia_porcentaje': asistencia.round(1),
        'ratio_creditos_aprobados': ratio_creditos.round(2),
        'materias_perdidas': materias_perdidas,
        'estrato': estrato,
        'financiamiento': financiamiento,
        'trabaja': trabaja,
        'horas_trabajo_semana': horas_trabajo,
        'horas_plataforma_semana': horas_plataforma.round(1),
        'interacciones_tutorias': interacciones_tutorias,
        'actividades_extracurriculares': actividades_extra,
        'edad': edad,
        'genero': genero,
        'distancia_campus_km': distancia_campus.round(1),
        'semestre': semestre
    })
    
    # ===============================
    # VARIABLE OBJETIVO (DESERCIÓN)
    # ===============================
    
    # Calcular score de riesgo basado en factores
    risk_score = np.zeros(n_students)
    
    # Factores académicos (más peso)
    risk_score += (5 - promedio_notas) * 0.15
    risk_score += (100 - asistencia) * 0.005
    risk_score += (1 - ratio_creditos) * 0.1
    risk_score += materias_perdidas * 0.05
    
    # Factores socioeconómicos
    risk_score += (7 - estrato) * 0.02
    risk_score += (financiamiento == 'propio').astype(float) * 0.1
    risk_score += horas_trabajo * 0.003
    
    # Factores de comportamiento
    risk_score += (20 - np.clip(horas_plataforma, 0, 20)) * 0.01
    risk_score += (10 - np.clip(interacciones_tutorias, 0, 10)) * 0.02
    
    # Convertir a probabilidad
    prob_dropout = 1 / (1 + np.exp(-risk_score + 1))
    prob_dropout = np.clip(prob_dropout, 0.05, 0.95)
    
    # Ajustar para tasa objetivo
    prob_dropout = prob_dropout * (dropout_rate / prob_dropout.mean())
    prob_dropout = np.clip(prob_dropout, 0, 1)
    
    # Generar deserciones
    df['desercion'] = np.random.binomial(1, prob_dropout)
    
    return df

# Generar dataset
print("="*60)
print("📊 GENERANDO DATOS DE ESTUDIANTES")
print("="*60)

df = generate_student_data(n_students=2000, dropout_rate=0.25)

print(f"\n✅ Dataset generado con {len(df)} estudiantes")
print(f"📈 Tasa de deserción: {df['desercion'].mean()*100:.1f}%")
print(f"\n📋 Variables del dataset ({df.shape[1]} columnas):")
print(df.columns.tolist())

In [ ]:
# Visualizar primeras filas y estadísticas
print("\n📋 Primeras 5 filas del dataset:")
display(df.head())

print("\n📊 Estadísticas descriptivas:")
display(df.describe())

## 📈 3. Análisis Exploratorio de Datos (EDA)

Antes de entrenar modelos, es crucial entender los datos. Este paso es fundamental en cualquier proyecto de ML (Campesato, 2020, p. 23).

In [ ]:
# Visualización de la distribución de clase objetivo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#2ECC71', '#E74C3C']
counts = df['desercion'].value_counts()

bars = axes[0].bar(['No Deserta', 'Deserta'], counts.values, color=colors, edgecolor='white', linewidth=2)
axes[0].set_ylabel('Cantidad de Estudiantes')
axes[0].set_title('Distribución de Clases - Deserción Estudiantil')

for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{count}', ha='center', fontsize=12, fontweight='bold')

# Gráfico circular
axes[1].pie(counts.values, labels=['No Deserta', 'Deserta'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=(0.05, 0.05), shadow=True)
axes[1].set_title('Proporción de Clases')

plt.suptitle('Análisis de Variable Objetivo - DESBALANCE DE CLASES', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n⚠️ DESBALANCE DE CLASES DETECTADO")
print(f"   Ratio: {counts.max()/counts.min():.2f}:1")
print(f"   Esto requiere técnicas de balanceo como SMOTE")

In [ ]:
# Distribución de variables numéricas
numeric_cols = ['promedio_notas', 'asistencia_porcentaje', 'ratio_creditos_aprobados',
                'materias_perdidas', 'horas_trabajo_semana', 'horas_plataforma_semana']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    
    # Histograma por clase
    for i, (label, color) in enumerate([('No Deserta', '#2ECC71'), ('Deserta', '#E74C3C')]):
        data = df[df['desercion'] == i][col]
        ax.hist(data, bins=20, alpha=0.6, label=label, color=color)
    
    ax.set_title(f'Distribución: {col}')
    ax.legend()
    ax.set_xlabel(col)

plt.suptitle('Distribución de Variables por Clase de Deserción', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
numeric_df = df.select_dtypes(include=[np.number]).drop('estudiante_id', axis=1)
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlación - Variables del Dataset', fontsize=14)
plt.tight_layout()
plt.show()

# Correlaciones con deserción
print("\n🔍 CORRELACIÓN CON DESERCIÓN:")
correlations = corr_matrix['desercion'].drop('desercion').sort_values(key=abs, ascending=False)
for var, corr in correlations.items():
    direction = "↑" if corr > 0 else "↓"
    print(f"   {var}: {corr:.3f} {direction}")

## 🔧 4. Componente de PREPROCESAMIENTO

### Componentes de ML: Limpieza y Transformación de Datos

El preprocesamiento incluye (Véliz Capuñay, 2020, pp. 33-50):

1. **Manejo de valores nulos** - Imputación
2. **Codificación de categóricas** - One-Hot Encoding
3. **Normalización** - StandardScaler
4. **Balanceo de clases** - SMOTE

In [ ]:
def preprocess_data(df, target_col='desercion', test_size=0.2, apply_smote=True):
    """
    Pipeline de preprocesamiento de datos.
    
    COMPONENTES DE PREPROCESAMIENTO:
    1. Separación features/target
    2. Codificación de variables categóricas
    3. División train/test estratificada
    4. Normalización
    5. Balanceo con SMOTE
    """
    print("="*60)
    print("🔧 PREPROCESAMIENTO DE DATOS")
    print("="*60)
    
    df_clean = df.copy()
    
    # 1. Eliminar ID
    if 'estudiante_id' in df_clean.columns:
        df_clean = df_clean.drop('estudiante_id', axis=1)
    
    # 2. Separar features y target
    y = df_clean[target_col].values
    X = df_clean.drop(target_col, axis=1)
    
    print(f"\n📊 Features originales: {X.shape[1]}")
    
    # 3. Identificar columnas categóricas y numéricas
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    
    print(f"   Numéricas: {len(num_cols)}")
    print(f"   Categóricas: {len(cat_cols)}")
    
    # 4. Codificación One-Hot de categóricas
    if cat_cols:
        print(f"\n🔄 Codificando variables categóricas: {cat_cols}")
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    
    feature_names = X.columns.tolist()
    print(f"   Features después de encoding: {len(feature_names)}")
    
    # 5. Convertir a array
    X = X.values
    
    # 6. División train/test estratificada
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    print(f"\n✂️ División Train/Test ({(1-test_size)*100:.0f}%/{test_size*100:.0f}%)")
    print(f"   Train: {X_train.shape[0]} muestras")
    print(f"   Test: {X_test.shape[0]} muestras")
    
    # 7. Normalización
    print(f"\n📏 Normalizando datos (StandardScaler)...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 8. Balanceo con SMOTE
    if apply_smote:
        print(f"\n⚖️ Aplicando SMOTE para balanceo de clases...")
        print(f"   Antes: {np.bincount(y_train)}")
        
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        
        print(f"   Después: {np.bincount(y_train)}")
    
    print(f"\n✅ Preprocesamiento completado")
    print(f"   X_train: {X_train.shape}")
    print(f"   X_test: {X_test.shape}")
    
    return X_train, X_test, y_train, y_test, feature_names, scaler

# Ejecutar preprocesamiento
X_train, X_test, y_train, y_test, feature_names, scaler = preprocess_data(df)

## 🤖 5. Componente de ENTRENAMIENTO DE MODELOS

### Componentes de ML: Algoritmos de Aprendizaje

Entrenaremos múltiples algoritmos para comparar rendimiento:

1. **Logistic Regression** - Modelo baseline interpretable
2. **Random Forest** - Ensemble de árboles, robusto
3. **Gradient Boosting** - Boosting secuencial
4. **XGBoost** - Estado del arte en datos tabulares

In [ ]:
def train_models(X_train, y_train, cv=5):
    """
    Entrena múltiples modelos de ML.
    
    COMPONENTES DE ENTRENAMIENTO:
    - Definición de modelos
    - Validación cruzada
    - Comparación de rendimiento
    """
    print("="*60)
    print("🤖 ENTRENAMIENTO DE MODELOS")
    print("="*60)
    
    models = {
        'Logistic Regression': LogisticRegression(
            max_iter=1000, random_state=42, class_weight='balanced'
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, max_depth=10, random_state=42,
            class_weight='balanced', n_jobs=-1
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, max_depth=5, random_state=42
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=42, eval_metric='logloss', use_label_encoder=False
        )
    }
    
    results = {}
    trained_models = {}
    
    for name, model in models.items():
        print(f"\n🔨 Entrenando {name}...")
        
        # Validación cruzada
        cv_scores = cross_val_score(
            model, X_train, y_train,
            cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
            scoring='f1'
        )
        
        # Entrenar modelo final
        model.fit(X_train, y_train)
        trained_models[name] = model
        
        results[name] = {
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'cv_scores': cv_scores
        }
        
        print(f"   F1-Score (CV): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    
    # Identificar mejor modelo
    best_model_name = max(results, key=lambda x: results[x]['cv_mean'])
    print(f"\n🏆 MEJOR MODELO: {best_model_name}")
    print(f"   F1-Score: {results[best_model_name]['cv_mean']:.4f}")
    
    return trained_models, results, best_model_name

# Entrenar modelos
trained_models, results, best_model_name = train_models(X_train, y_train)

In [ ]:
# Visualizar comparación de modelos
model_names = list(results.keys())
cv_means = [results[m]['cv_mean'] for m in model_names]
cv_stds = [results[m]['cv_std'] for m in model_names]

# Ordenar por rendimiento
sorted_indices = np.argsort(cv_means)
model_names = [model_names[i] for i in sorted_indices]
cv_means = [cv_means[i] for i in sorted_indices]
cv_stds = [cv_stds[i] for i in sorted_indices]

# Gráfico
plt.figure(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(model_names)))

bars = plt.barh(model_names, cv_means, xerr=cv_stds, color=colors,
                edgecolor='white', linewidth=2, capsize=5)

plt.xlabel('F1-Score (Validación Cruzada)')
plt.title('Comparación de Modelos de Machine Learning', fontsize=14)

for bar, mean in zip(bars, cv_means):
    plt.text(mean + 0.01, bar.get_y() + bar.get_height()/2,
             f'{mean:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 📊 6. Componente de EVALUACIÓN

### Componentes de ML: Métricas y Validación

Evaluamos el mejor modelo con métricas relevantes para el problema:

- **Accuracy**: % de predicciones correctas
- **Precision**: De los predichos como desertores, % que realmente desertan
- **Recall**: De los desertores reales, % que detectamos (MUY IMPORTANTE)
- **F1-Score**: Balance entre precision y recall
- **AUC-ROC**: Capacidad discriminativa del modelo

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Evaluación completa del modelo.
    
    COMPONENTES DE EVALUACIÓN:
    - Métricas de clasificación
    - Matriz de confusión
    - Curvas ROC y PR
    """
    print("="*60)
    print(f"📊 EVALUACIÓN: {model_name}")
    print("="*60)
    
    # Predicciones
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Calcular métricas
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_proba)
    }
    
    # Imprimir métricas
    print(f"\n🎯 MÉTRICAS DE CLASIFICACIÓN")
    print(f"   Accuracy:    {metrics['accuracy']:.4f}  (% predicciones correctas)")
    print(f"   Precision:   {metrics['precision']:.4f}  (% de predichos positivos correctos)")
    print(f"   Recall:      {metrics['recall']:.4f}  (% de positivos reales detectados)")
    print(f"   F1-Score:    {metrics['f1_score']:.4f}  (balance precision-recall)")
    print(f"   AUC-ROC:     {metrics['auc_roc']:.4f}  (capacidad discriminativa)")
    
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n📋 MATRIZ DE CONFUSIÓN")
    print(f"   TN={tn} (correctos no desertores)  │ FP={fp} (falsas alarmas)")
    print(f"   FN={fn} (desertores NO detectados) │ TP={tp} (desertores detectados)")
    
    # Interpretación
    print(f"\n💡 INTERPRETACIÓN")
    if metrics['recall'] >= 0.7:
        print(f"   ✅ Buen recall: detectamos {metrics['recall']*100:.1f}% de desertores")
    else:
        print(f"   ⚠️ Recall bajo: {fn} estudiantes en riesgo NO fueron detectados")
    
    return metrics, y_pred, y_proba

# Evaluar mejor modelo
best_model = trained_models[best_model_name]
metrics, y_pred, y_proba = evaluate_model(best_model, X_test, y_test, best_model_name)

In [ ]:
# Visualizaciones de evaluación
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Deserta', 'Deserta'],
            yticklabels=['No Deserta', 'Deserta'],
            ax=axes[0, 0])
axes[0, 0].set_title('Matriz de Confusión')
axes[0, 0].set_ylabel('Valor Real')
axes[0, 0].set_xlabel('Predicción')

# 2. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[0, 1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc:.3f}')
axes[0, 1].plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Aleatorio')
axes[0, 1].fill_between(fpr, tpr, alpha=0.3)
axes[0, 1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0, 1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[0, 1].set_title('Curva ROC')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Curva Precision-Recall
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
axes[1, 0].plot(recall_curve, precision_curve, 'b-', linewidth=2)
axes[1, 0].fill_between(recall_curve, precision_curve, alpha=0.3)
axes[1, 0].axhline(y=y_test.mean(), color='r', linestyle='--', 
                    label=f'Baseline ({y_test.mean():.2f})')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Curva Precision-Recall')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Distribución de probabilidades
axes[1, 1].hist(y_proba[y_test == 0], bins=30, alpha=0.6, 
                label='No Deserta', color='#2ECC71')
axes[1, 1].hist(y_proba[y_test == 1], bins=30, alpha=0.6, 
                label='Deserta', color='#E74C3C')
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', label='Umbral (0.5)')
axes[1, 1].set_xlabel('Probabilidad de Deserción')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].set_title('Distribución de Probabilidades por Clase')
axes[1, 1].legend()

plt.suptitle(f'Dashboard de Evaluación - {best_model_name}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 🔍 7. Interpretabilidad: Importancia de Características

### Componente de ML: Explicabilidad (XAI)

Entender QUÉ factores influyen en las predicciones es crucial para:
1. Validar que el modelo tiene sentido
2. Diseñar intervenciones efectivas
3. Comunicar resultados a stakeholders

In [ ]:
# Obtener importancia de características (para modelos que lo soporten)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    
    # Crear DataFrame ordenado
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=True)
    
    # Visualizar top 15
    top_n = 15
    df_plot = importance_df.tail(top_n)
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(df_plot)))
    
    bars = plt.barh(df_plot['feature'], df_plot['importance'], color=colors)
    plt.xlabel('Importancia')
    plt.title(f'Top {top_n} Características más Importantes - {best_model_name}', fontsize=14)
    
    for bar, val in zip(bars, df_plot['importance']):
        plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 TOP 10 FACTORES MÁS IMPORTANTES PARA PREDECIR DESERCIÓN:")
    for i, row in importance_df.tail(10).iloc[::-1].iterrows():
        print(f"   {row['feature']}: {row['importance']:.4f}")

## 🚨 8. Componente de SALIDA: Sistema de Alertas Tempranas

### Componentes de ML: Outputs y Acciones

El modelo genera:
1. **Score de Riesgo** (0-100%)
2. **Clasificación por nivel** (Bajo, Medio, Alto, Crítico)
3. **Alertas tempranas** para intervención
4. **Recomendaciones** personalizadas

In [ ]:
def generate_alerts(df_original, X_test_scaled, y_proba, test_indices):
    """
    Genera alertas tempranas basadas en las predicciones.
    
    COMPONENTES DE SALIDA:
    - Score de riesgo porcentual
    - Clasificación por nivel
    - Factores de riesgo identificados
    - Recomendaciones de intervención
    """
    print("="*60)
    print("🚨 SISTEMA DE ALERTAS TEMPRANAS")
    print("="*60)
    
    # Clasificar por nivel de riesgo
    def classify_risk(score):
        if score < 0.25:
            return 'BAJO'
        elif score < 0.50:
            return 'MEDIO'
        elif score < 0.75:
            return 'ALTO'
        else:
            return 'CRÍTICO'
    
    # Crear DataFrame de alertas
    alerts_df = pd.DataFrame({
        'estudiante_id': test_indices + 1,
        'risk_score': (y_proba * 100).round(1),
        'risk_level': [classify_risk(p) for p in y_proba]
    })
    
    # Calcular prioridad
    def get_priority(level):
        return {'CRÍTICO': 1, 'ALTO': 2, 'MEDIO': 3, 'BAJO': 4}[level]
    
    alerts_df['priority'] = alerts_df['risk_level'].apply(get_priority)
    
    # Estadísticas
    risk_counts = alerts_df['risk_level'].value_counts()
    total = len(alerts_df)
    
    print(f"\n📊 DISTRIBUCIÓN DE RIESGO:")
    for level in ['CRÍTICO', 'ALTO', 'MEDIO', 'BAJO']:
        count = risk_counts.get(level, 0)
        pct = count / total * 100
        emoji = {'CRÍTICO': '🔴', 'ALTO': '🟠', 'MEDIO': '🟡', 'BAJO': '🟢'}[level]
        print(f"   {emoji} {level}: {count} estudiantes ({pct:.1f}%)")
    
    high_risk = len(alerts_df[alerts_df['risk_level'].isin(['ALTO', 'CRÍTICO'])])
    print(f"\n⚠️ REQUIEREN INTERVENCIÓN URGENTE: {high_risk} estudiantes")
    
    return alerts_df

# Generar alertas para el conjunto de prueba
test_indices = np.arange(len(y_test))
alerts_df = generate_alerts(df, X_test, y_proba, test_indices)

# Mostrar estudiantes de mayor riesgo
print("\n🚨 TOP 10 ESTUDIANTES EN MAYOR RIESGO:")
display(alerts_df.nlargest(10, 'risk_score')[['estudiante_id', 'risk_score', 'risk_level', 'priority']])

In [ ]:
# Visualización del sistema de alertas
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Colores por nivel de riesgo
risk_colors = {'BAJO': '#27AE60', 'MEDIO': '#F1C40F', 'ALTO': '#E67E22', 'CRÍTICO': '#C0392B'}

# 1. Distribución por nivel
counts = alerts_df['risk_level'].value_counts().reindex(['BAJO', 'MEDIO', 'ALTO', 'CRÍTICO'], fill_value=0)
bars = axes[0].bar(counts.index, counts.values,
                   color=[risk_colors[r] for r in counts.index],
                   edgecolor='white', linewidth=2)
axes[0].set_title('Estudiantes por Nivel de Riesgo')
axes[0].set_ylabel('Cantidad')

for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(count), ha='center', fontweight='bold')

# 2. Histograma de scores
axes[1].hist(alerts_df['risk_score'], bins=25, color='#3498DB',
             edgecolor='white', alpha=0.7)
axes[1].axvline(50, color='red', linestyle='--', label='Umbral 50%')
axes[1].axvline(alerts_df['risk_score'].mean(), color='green',
                linestyle=':', label=f'Media: {alerts_df["risk_score"].mean():.1f}%')
axes[1].set_title('Distribución de Scores de Riesgo')
axes[1].set_xlabel('Score de Riesgo (%)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

# 3. Por prioridad
priority_counts = alerts_df['priority'].value_counts().sort_index()
priority_colors = ['#C0392B', '#E67E22', '#F1C40F', '#27AE60']
axes[2].bar(priority_counts.index, priority_counts.values,
            color=priority_colors[:len(priority_counts)],
            edgecolor='white', linewidth=2)
axes[2].set_title('Estudiantes por Prioridad de Atención')
axes[2].set_xlabel('Prioridad (1=Urgente, 4=Baja)')
axes[2].set_ylabel('Cantidad')

plt.suptitle('Dashboard del Sistema de Alertas Tempranas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 📋 9. Reporte Final de Intervención

Generamos un reporte ejecutivo para el equipo de bienestar estudiantil.

In [ ]:
def generate_intervention_report(alerts_df, metrics, best_model_name):
    """
    Genera reporte ejecutivo de intervención.
    """
    total = len(alerts_df)
    critico = len(alerts_df[alerts_df['risk_level'] == 'CRÍTICO'])
    alto = len(alerts_df[alerts_df['risk_level'] == 'ALTO'])
    medio = len(alerts_df[alerts_df['risk_level'] == 'MEDIO'])
    bajo = len(alerts_df[alerts_df['risk_level'] == 'BAJO'])
    
    report = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         REPORTE DE INTERVENCIÓN - SISTEMA DE ALERTAS TEMPRANAS               ║
║                     PREDICCIÓN DE DESERCIÓN ESTUDIANTIL                      ║
╚══════════════════════════════════════════════════════════════════════════════╝

📊 RESUMEN EJECUTIVO
────────────────────────────────────────────────────────────────────────────────
  Total estudiantes analizados:    {total:>5}
  Modelo utilizado:                {best_model_name}
  
  Distribución por nivel de riesgo:
  ┌─────────────┬──────────┬────────────┐
  │ Nivel       │ Cantidad │ Porcentaje │
  ├─────────────┼──────────┼────────────┤
  │ 🔴 CRÍTICO  │ {critico:>8} │ {critico/total*100:>9.1f}% │
  │ 🟠 ALTO     │ {alto:>8} │ {alto/total*100:>9.1f}% │
  │ 🟡 MEDIO    │ {medio:>8} │ {medio/total*100:>9.1f}% │
  │ 🟢 BAJO     │ {bajo:>8} │ {bajo/total*100:>9.1f}% │
  └─────────────┴──────────┴────────────┘

📈 MÉTRICAS DEL MODELO
────────────────────────────────────────────────────────────────────────────────
  Accuracy:       {metrics['accuracy']:.4f}   │ Precision:  {metrics['precision']:.4f}
  Recall:         {metrics['recall']:.4f}   │ F1-Score:   {metrics['f1_score']:.4f}
  AUC-ROC:        {metrics['auc_roc']:.4f}   │

⚠️ ACCIONES REQUERIDAS
────────────────────────────────────────────────────────────────────────────────
  • {critico + alto} estudiantes requieren intervención URGENTE
  • Convocar reunión del comité de permanencia estudiantil
  • Asignar tutores para casos críticos (ratio 1:5 máximo)
  • Programar contacto telefónico con estudiantes críticos en 48h

💡 RECOMENDACIONES GENERALES
────────────────────────────────────────────────────────────────────────────────
  1. Priorizar estudiantes con score > 75%
  2. Vincular estudiantes en riesgo con servicios de bienestar
  3. Implementar programa de tutorías académicas
  4. Evaluar opciones de financiamiento para casos socioeconómicos
  5. Monitorear asistencia semanal de estudiantes en riesgo

═══════════════════════════════════════════════════════════════════════════════
  Este reporte fue generado por el Sistema de Predicción de Deserción
  Estudiantil basado en Machine Learning - UNIMINUTO
═══════════════════════════════════════════════════════════════════════════════
"""
    return report

# Generar e imprimir reporte
report = generate_intervention_report(alerts_df, metrics, best_model_name)
print(report)

## 🎓 10. Conclusiones y Respuestas del Foro

### Pregunta 1: ¿Por qué es importante considerar la cantidad de datos en ML?

Como demostramos en este proyecto:

1. **Generalización**: Usamos 2000 estudiantes para que el modelo aprenda patrones robustos
2. **Validación cruzada**: Pudimos dividir en 5 folds sin perder representatividad
3. **Desbalance**: SMOTE requiere suficientes ejemplos de la clase minoritaria
4. **Train/Test split**: División 80/20 mantuvo suficientes casos en cada conjunto

### Pregunta 2: ¿Cuáles son los componentes de Machine Learning?

| Componente | Implementación en este proyecto |
|------------|--------------------------------|
| **Datos de Entrada** | Notas, asistencia, socioeconómicos, comportamiento |
| **Preprocesamiento** | Encoding, normalización, SMOTE |
| **Algoritmos** | Random Forest, XGBoost, Logistic Regression |
| **Validación** | Cross-validation, métricas de evaluación |
| **Salidas** | Scores de riesgo, alertas, recomendaciones |

### Mejoras Propuestas

1. **MLOps**: Implementar monitoreo continuo del modelo en producción
2. **XAI**: Usar SHAP para explicaciones más detalladas
3. **Feedback Loop**: Incorporar resultados reales para reentrenamiento
4. **Dashboard**: Crear interfaz web para visualización en tiempo real

## 📚 Referencias Bibliográficas

- Barrero Ortiz, G. (2020). *Machine Learning: 50 Conceptos Clave para Entenderlo* (pp. 16, 56, 70). Paradigma.
- Rothman, D. (2018). *Artificial intelligence by example* (pp. 46-58). Packt Publishing.
- Véliz Capuñay, C. (2020). *Aprendizaje automático: Introducción al aprendizaje profundo* (pp. 33-105). PUCP.
- Campesato, O. (2020). *AI, ML, and Deep Learning* (pp. 18-49). Mercury Learning.

---

*Proyecto desarrollado para el Foro de Componentes de Machine Learning - UNIMINUTO - 2025*